# Akili MuJoCo v0.2 — Three-Seed Publication Rerun

Runs the existing CPU flagship for seeds 1, 2 and 3. Completed seeds are skipped on restart. Model checkpoints are excluded from the downloaded evidence ZIP.

In [ ]:
import os, sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nbformat", "nbclient", "nbconvert"],
    check=True,
)
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount(os.getenv("AKILI_BATCH_DRIVE_MOUNT", "/content/drive"))
else:
    print("Not running in Colab; Drive mount skipped.")


In [ ]:
import ast, importlib, json, shutil, sys
from pathlib import Path

RUNNER_NAME = "akili_publication_batch_runner_v1"
RUNNER_SOURCE = '\nfrom __future__ import annotations\n\nimport csv\nimport datetime as dt\nimport json\nimport os\nimport subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Mapping, Optional, Sequence\n\n\nPROTOCOL = "akili-publication-notebook-batch-runner-v1"\n\n\ndef utc_now() -> str:\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef read_json(path: Path) -> Optional[Dict[str, Any]]:\n    try:\n        value = json.loads(path.read_text(encoding="utf-8"))\n        return value if isinstance(value, dict) else None\n    except Exception:\n        return None\n\n\ndef hard_checks_status(run_root: Optional[Path]) -> Optional[bool]:\n    if run_root is None:\n        return None\n    path = run_root / "hard_checks.json"\n    payload = read_json(path) if path.is_file() else None\n    if payload is None:\n        return None\n    if isinstance(payload.get("all_passed"), bool):\n        return bool(payload["all_passed"])\n    bool_values = [value for value in payload.values() if isinstance(value, bool)]\n    return all(bool_values) if bool_values else None\n\n\ndef newest_run(output_root: Path) -> Optional[Path]:\n    runs = sorted(\n        (path for path in output_root.glob("run_*") if path.is_dir()),\n        key=lambda path: path.stat().st_mtime,\n        reverse=True,\n    )\n    return runs[0] if runs else None\n\n\ndef execute_notebook(\n    source_notebook: Path,\n    executed_notebook: Path,\n    *,\n    environment: Mapping[str, str],\n    log_path: Path,\n    timeout_seconds: int = 0,\n) -> int:\n    executed_notebook.parent.mkdir(parents=True, exist_ok=True)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    command = [\n        sys.executable,\n        "-m",\n        "jupyter",\n        "nbconvert",\n        "--to",\n        "notebook",\n        "--execute",\n        str(source_notebook),\n        "--output",\n        str(executed_notebook),\n        "--ExecutePreprocessor.kernel_name=python3",\n        f"--ExecutePreprocessor.timeout={timeout_seconds}",\n    ]\n    env = dict(os.environ)\n    env.update({key: str(value) for key, value in environment.items()})\n    with log_path.open("w", encoding="utf-8") as log:\n        process = subprocess.run(\n            command,\n            env=env,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            check=False,\n        )\n    return int(process.returncode)\n\n\ndef evidence_zip(batch_root: Path, destination: Path) -> None:\n    excluded_suffixes = {".safetensors", ".pt", ".pth", ".bin", ".ckpt"}\n    excluded_parts = {"checkpoints", "__pycache__", ".ipynb_checkpoints"}\n    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(item for item in batch_root.rglob("*") if item.is_file()):\n            if path.suffix.lower() in excluded_suffixes:\n                continue\n            if any(part in excluded_parts for part in path.parts):\n                continue\n            archive.write(path, path.relative_to(batch_root).as_posix())\n\n\ndef run_seed_batch(\n    *,\n    source_notebook: Path,\n    batch_root: Path,\n    seeds: Sequence[int],\n    env_builder,\n    output_root_builder,\n    force: bool = False,\n) -> Dict[str, Any]:\n    batch_root.mkdir(parents=True, exist_ok=True)\n    records: List[Dict[str, Any]] = []\n\n    for seed in seeds:\n        seed_root = batch_root / f"seed_{seed}"\n        seed_root.mkdir(parents=True, exist_ok=True)\n        completion_path = seed_root / "COMPLETE.json"\n        existing = read_json(completion_path)\n        if (\n            not force\n            and existing is not None\n            and existing.get("returncode") == 0\n            and existing.get("run_root")\n            and Path(str(existing["run_root"])).is_dir()\n        ):\n            records.append(existing)\n            print(f"[resume] seed={seed} already complete")\n            continue\n\n        output_root = output_root_builder(seed)\n        output_root.mkdir(parents=True, exist_ok=True)\n        executed_notebook = seed_root / f"executed_seed_{seed}.ipynb"\n        log_path = seed_root / f"seed_{seed}.log"\n        environment = env_builder(seed, output_root)\n        print(f"[run] seed={seed} output={output_root}")\n        returncode = execute_notebook(\n            source_notebook,\n            executed_notebook,\n            environment=environment,\n            log_path=log_path,\n            timeout_seconds=0,\n        )\n        run_root = newest_run(output_root)\n        all_passed = hard_checks_status(run_root)\n        record = {\n            "protocol": PROTOCOL,\n            "seed": seed,\n            "started_output_root": str(output_root),\n            "run_root": str(run_root) if run_root else None,\n            "returncode": returncode,\n            "all_passed": all_passed,\n            "executed_notebook": str(executed_notebook),\n            "log": str(log_path),\n            "completed_at": utc_now(),\n        }\n        completion_path.write_text(\n            json.dumps(record, indent=2, sort_keys=True), encoding="utf-8"\n        )\n        records.append(record)\n        if returncode != 0:\n            raise RuntimeError(\n                f"Seed {seed} failed. Read {log_path}; completed seeds remain resumable."\n            )\n\n    aggregate = {\n        "protocol": PROTOCOL,\n        "created_at": utc_now(),\n        "seeds": list(seeds),\n        "records": records,\n        "all_processes_completed": all(record["returncode"] == 0 for record in records),\n        "all_hard_checks_passed": all(record["all_passed"] is True for record in records),\n    }\n    with (batch_root / "BATCH_SUMMARY.csv").open(\n        "w", encoding="utf-8", newline=""\n    ) as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=["seed", "returncode", "all_passed", "run_root", "log"],\n        )\n        writer.writeheader()\n        for record in records:\n            writer.writerow({key: record.get(key) for key in writer.fieldnames})\n\n    zip_path = batch_root.parent / f"{batch_root.name}_evidence.zip"\n    evidence_zip(batch_root, zip_path)\n    aggregate["evidence_zip"] = str(zip_path)\n    (batch_root / "BATCH_SUMMARY.json").write_text(\n        json.dumps(aggregate, indent=2, sort_keys=True), encoding="utf-8"\n    )\n    return aggregate\n\n\ndef synthetic_verification(root: Path) -> Dict[str, Any]:\n    root.mkdir(parents=True, exist_ok=True)\n    fake_run = root / "runs" / "run_test"\n    fake_run.mkdir(parents=True, exist_ok=True)\n    (fake_run / "hard_checks.json").write_text(\n        \'{"all_passed": true}\', encoding="utf-8"\n    )\n    checks = {\n        "newest_run_detected": newest_run(root / "runs") == fake_run,\n        "hard_checks_read": hard_checks_status(fake_run) is True,\n    }\n    return {"passed": all(checks.values()), "checks": checks}\n'
SOURCE_NOTEBOOK_TEXT = '{\n "cells": [\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# NOTEBOOK CELLS for Akili Robotics v0.2 MuJoCo — assembled into .ipynb after verification\\n",\n    "# marker format:  # %% [md]  or  # %%"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# AKILI ROBOTICS v0.2 — *\\"Four systems, one survivor, then beyond\\"*\\n",\n    "**Standalone notebook — MuJoCo 3D arm, CPU-only (Colab or a Mac).**\\n",\n    "#\\n",\n    "A 3-DOF robot arm in real 3D-rendered physics learns skills sequentially:\\n",\n    "reach two targets, press a button, and route around a hazard column.\\n",\n    "Then a **poisoned skill** (inverted joint commands — the flipped-actuator nightmare)\\n",\n    "and a **poisoned skill update** arrive. Four systems face the same world, same data, same poison:\\n",\n    "#\\n",\n    "1. **Sequential fine-tune** — one shared model: forgets old skills on camera, poison baked in\\n",\n    "2. **Motion-library retrieval** (the robotics RAG) — replays recordings, dies on a 5 cm shift\\n",\n    "3. **Naive adapter bank** — modularity without governance: ships the poison to production\\n",\n    "4. **Akili** — frozen trunk + write-once adapters + validation gates + versioned rollback\\n",\n    "#\\n",\n    "Then Akili goes *beyond* survival:\\n",\n    "- **Act 4** — a poisoned *skill update* (v2 drops the hazard routing) is shadow-tested and rejected while v1 keeps working: zero downtime\\n",\n    "- **Act 5** — certified skill *composition*: a chained mission with a cryptographic receipt per step\\n",\n    "#\\n",\n    "**Runtime:** ~15–25 min on CPU. Fully resumable (`AKILI_ARM_RESUME=latest`).\\n",\n    "Everything (checkpoints, registry, audit chain, report, GIFs) lands in one run folder.\\n",\n    "Control is kinematic in MuJoCo (full 3D collision geometry + 3D rendering); training is CPU torch."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 1 — CONFIG\\n",\n    "# ============================================================\\n",\n    "import os, json\\n",\n    "\\n",\n    "def _env(n, d):\\n",\n    "    v = os.environ.get(n, \\"\\")\\n",\n    "    return v if str(v).strip() else d\\n",\n    "\\n",\n    "CONFIG = {\\n",\n    "    \\"seed\\": int(_env(\\"AKILI_ARM_SEED\\", \\"1\\")),\\n",\n    "    \\"out_dir\\": _env(\\"AKILI_ARM_OUT\\", \\"/content/drive/MyDrive/AKM_CLR/stage05/akili_robotics_v0_2_mujoco\\" if\\n",\n    "                    os.path.isdir(\\"/content/drive\\") else os.path.expanduser(\\"~/akili_robotics_v0_2\\")),\\n",\n    "    \\"resume\\": _env(\\"AKILI_ARM_RESUME\\", \\"\\"),\\n",\n    "    \\"steps\\": 100, \\"dt\\": 0.05, \\"success_dist\\": 0.025, \\"qdot_max\\": 1.2,\\n",\n    "    \\"hidden\\": 128, \\"adapter_rank\\": 8,\\n",\n    "    \\"train\\": {\\"epochs\\": 70, \\"lr\\": 2e-3, \\"batch\\": 256, \\"eps_per_skill\\": 300},\\n",\n    "    \\"eval_episodes\\": 50,\\n",\n    "    \\"activation_min_success\\": 0.85,\\n",\n    "    \\"skills\\": [\\"reach_A\\", \\"reach_B\\", \\"press_button\\", \\"avoid_zone\\"],\\n",\n    "    \\"poison_skill\\": \\"reach_C\\",\\n",\n    "    \\"update_skill\\": \\"avoid_zone\\",     # Act 4: the update that drops hazard routing\\n",\n    "    \\"v2_poison_frac\\": 0.35,          # fraction of corner-cutting trajectories in the v2 candidate\\n",\n    "}\\n",\n    "print(json.dumps(CONFIG, indent=2))"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 2 — IMPORTS, GL SETUP, DETERMINISM, RUN DIR\\n",\n    "# ============================================================\\n",\n    "import os, json, glob, math, time, hashlib, datetime, random\\n",\n    "import numpy as np\\n",\n    "\\n",\n    "# --- dependency guard (Colab: mujoco is not preinstalled) ---\\n",\n    "import sys, subprocess\\n",\n    "try:\\n",\n    "    import mujoco  # noqa\\n",\n    "except ImportError:\\n",\n    "    print(\\"[env] installing mujoco, imageio, pillow...\\")\\n",\n    "    subprocess.run([sys.executable, \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", \\"mujoco\\", \\"imageio\\", \\"pillow\\"],\\n",\n    "                   check=False)\\n",\n    "\\n",\n    "# --- GL backend selection MUST happen before importing mujoco ---\\n",\n    "# order: macOS/Windows native -> DISPLAY (Linux desktop) -> EGL (Colab/headless) -> osmesa\\n",\n    "import ctypes.util\\n",\n    "GL_MODE = None\\n",\n    "if sys.platform == \\"darwin\\" or sys.platform.startswith(\\"win\\"):\\n",\n    "    GL_MODE = \\"native\\"                 # mujoco uses CGL/WGL offscreen contexts\\n",\n    "elif os.environ.get(\\"DISPLAY\\"):\\n",\n    "    GL_MODE = \\"glfw\\"\\n",\n    "elif ctypes.util.find_library(\\"EGL\\"):\\n",\n    "    os.environ[\\"MUJOCO_GL\\"] = \\"egl\\"; GL_MODE = \\"egl\\"\\n",\n    "elif ctypes.util.find_library(\\"OSMesa\\"):\\n",\n    "    os.environ[\\"MUJOCO_GL\\"] = \\"osmesa\\"; GL_MODE = \\"osmesa\\"\\n",\n    "\\n",\n    "import torch\\n",\n    "import torch.nn as nn\\n",\n    "import matplotlib\\n",\n    "matplotlib.use(\\"Agg\\")\\n",\n    "import matplotlib.pyplot as plt\\n",\n    "\\n",\n    "MUJOCO_OK = True\\n",\n    "try:\\n",\n    "    import mujoco\\n",\n    "    import imageio.v2 as imageio\\n",\n    "except Exception as e:\\n",\n    "    MUJOCO_OK = False\\n",\n    "    print(\\"[env] mujoco unavailable:\\", e)\\n",\n    "\\n",\n    "SEED = CONFIG[\\"seed\\"]\\n",\n    "random.seed(SEED); np.random.seed(SEED)\\n",\n    "torch.manual_seed(SEED)\\n",\n    "torch.use_deterministic_algorithms(True, warn_only=True)\\n",\n    "print(f\\"[env] torch={torch.__version__} mujoco={mujoco.__version__ if MUJOCO_OK else \'n/a\'} GL={GL_MODE}\\")\\n",\n    "\\n",\n    "def resolve_out():\\n",\n    "    if CONFIG[\\"resume\\"] and CONFIG[\\"resume\\"] != \\"latest\\":\\n",\n    "        assert os.path.isdir(CONFIG[\\"resume\\"])\\n",\n    "        return CONFIG[\\"resume\\"]\\n",\n    "    if CONFIG[\\"resume\\"] == \\"latest\\":\\n",\n    "        runs = sorted(glob.glob(os.path.join(CONFIG[\\"out_dir\\"], \\"run_*\\")))\\n",\n    "        assert runs, f\\"[resume] no runs under {CONFIG[\'out_dir\']}\\"\\n",\n    "        return runs[-1]\\n",\n    "    return os.path.join(CONFIG[\\"out_dir\\"],\\n",\n    "                        \\"run_\\" + datetime.datetime.now(datetime.timezone.utc).strftime(\\"%Y%m%dT%H%M%SZ\\"))\\n",\n    "\\n",\n    "RUN_DIR = resolve_out()\\n",\n    "os.makedirs(RUN_DIR, exist_ok=True)\\n",\n    "CKPT = os.path.join(RUN_DIR, \\"checkpoints\\"); os.makedirs(CKPT, exist_ok=True)\\n",\n    "ANIM = os.path.join(RUN_DIR, \\"animations\\"); os.makedirs(ANIM, exist_ok=True)\\n",\n    "print(f\\"[run] {RUN_DIR}\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 3 — THE WORLD: 3-DOF SCARA ARM, HAZARD COLUMN, TARGETS\\n",\n    "# ============================================================\\n",\n    "SCENE_XML = \\"\\"\\"\\n",\n    "<mujoco model=\\"akili_arm\\">\\n",\n    "  <option timestep=\\"0.01\\"/>\\n",\n    "  <worldbody>\\n",\n    "    <light pos=\\"0.3 -0.3 1.4\\" dir=\\"-0.15 0.25 -1\\" diffuse=\\"0.9 0.9 0.9\\" specular=\\"0.25 0.25 0.25\\"/>\\n",\n    "    <geom name=\\"table\\" type=\\"plane\\" pos=\\"0 0 0\\" size=\\"1.2 1.2 0.1\\" rgba=\\"0.86 0.88 0.91 1\\"/>\\n",\n    "    <geom name=\\"hazard\\" type=\\"cylinder\\" pos=\\"0.30 0.08 0.10\\" size=\\"0.07 0.10\\" rgba=\\"0.88 0.10 0.10 0.92\\"/>\\n",\n    "    <geom name=\\"button\\" type=\\"cylinder\\" pos=\\"0.10 -0.25 0.015\\" size=\\"0.035 0.015\\" rgba=\\"1.0 0.78 0.10 1\\"/>\\n",\n    "    <body name=\\"base\\" pos=\\"-0.12 -0.08 0.03\\">\\n",\n    "      <geom type=\\"cylinder\\" size=\\"0.05 0.03\\" rgba=\\"0.22 0.22 0.28 1\\"/>\\n",\n    "      <body name=\\"link1\\" pos=\\"0 0 0.035\\">\\n",\n    "        <joint name=\\"j1\\" type=\\"hinge\\" axis=\\"0 0 1\\" range=\\"-2.7 2.7\\" damping=\\"0.02\\"/>\\n",\n    "        <geom type=\\"capsule\\" fromto=\\"0 0 0 0.25 0 0\\" size=\\"0.030\\" rgba=\\"0.16 0.38 0.80 1\\"/>\\n",\n    "        <body name=\\"link2\\" pos=\\"0.25 0 0\\">\\n",\n    "          <joint name=\\"j2\\" type=\\"hinge\\" axis=\\"0 0 1\\" range=\\"-2.7 2.7\\" damping=\\"0.02\\"/>\\n",\n    "          <geom type=\\"capsule\\" fromto=\\"0 0 0 0.20 0 0\\" size=\\"0.028\\" rgba=\\"0.20 0.48 0.86 1\\"/>\\n",\n    "          <body name=\\"link3\\" pos=\\"0.20 0 0\\">\\n",\n    "            <joint name=\\"j3\\" type=\\"hinge\\" axis=\\"0 0 1\\" range=\\"-2.7 2.7\\" damping=\\"0.02\\"/>\\n",\n    "            <geom type=\\"capsule\\" fromto=\\"0 0 0 0.15 0 0\\" size=\\"0.026\\" rgba=\\"0.30 0.58 0.92 1\\"/>\\n",\n    "            <geom name=\\"eeball\\" type=\\"sphere\\" pos=\\"0.15 0 0\\" size=\\"0.024\\" rgba=\\"0.95 0.25 0.20 1\\"/>\\n",\n    "            <site name=\\"ee\\" pos=\\"0.15 0 0\\" size=\\"0.006\\"/>\\n",\n    "          </body>\\n",\n    "        </body>\\n",\n    "      </body>\\n",\n    "    </body>\\n",\n    "    <body name=\\"goal_marker\\" mocap=\\"true\\" pos=\\"0 0 0.02\\">\\n",\n    "      <geom type=\\"sphere\\" size=\\"0.022\\" rgba=\\"0.10 0.75 0.25 0.9\\" contype=\\"0\\" conaffinity=\\"0\\"/>\\n",\n    "    </body>\\n",\n    "    <body name=\\"wp_marker\\" mocap=\\"true\\" pos=\\"0 0 0.02\\">\\n",\n    "      <geom type=\\"sphere\\" size=\\"0.014\\" rgba=\\"0.10 0.75 0.90 0.9\\" contype=\\"0\\" conaffinity=\\"0\\"/>\\n",\n    "    </body>\\n",\n    "  </worldbody>\\n",\n    "</mujoco>\\n",\n    "\\"\\"\\"\\n",\n    "\\n",\n    "model = mujoco.MjModel.from_xml_string(SCENE_XML)\\n",\n    "data = mujoco.MjData(model)\\n",\n    "scratch = mujoco.MjData(model)          # private state for expert/FK — never touches live data\\n",\n    "\\n",\n    "EE_SITE = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, \\"ee\\")\\n",\n    "LINK2_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, \\"link2\\")\\n",\n    "LINK3_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, \\"link3\\")\\n",\n    "GOAL_MID = int(model.body_mocapid[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, \\"goal_marker\\")])\\n",\n    "WP_MID = int(model.body_mocapid[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, \\"wp_marker\\")])\\n",\n    "\\n",\n    "BASE = np.array([-0.12, -0.08])\\n",\n    "HAZARD = {\\"c\\": np.array([0.30, 0.08]), \\"r\\": 0.07, \\"margin\\": 0.04}     # margin ~ capsule radii\\n",\n    "RM = HAZARD[\\"r\\"] + HAZARD[\\"margin\\"]                                   # 0.11 no-contact radius\\n",\n    "TARGETS = {\\"reach_A\\": np.array([-0.30, 0.28]), \\"reach_B\\": np.array([0.33, 0.26]),\\n",\n    "           \\"reach_C\\": np.array([-0.05, 0.45])}\\n",\n    "BUTTON = np.array([0.10, -0.25])\\n",\n    "AVOID_GOAL = np.array([0.34, -0.22])\\n",\n    "AVOID_WP = np.array([0.006, -0.09])                                   # certified dogleg corner\\n",\n    "PRESS_WP = np.array([0.0, -0.15])                                     # press approach corner (keeps the sweep clear)\\n",\n    "Q_HOME = np.array([0.8, 0.8, -0.7])\\n",\n    "VMAX_EE, QDOT_MAX, LAM = 0.22, CONFIG[\\"qdot_max\\"], 0.05\\n",\n    "CLEAR = 0.05\\n",\n    "REACT_LINK, GAIN_LINK = RM + 0.15, 6.0\\n",\n    "\\n",\n    "def reset_to(q):\\n",\n    "    mujoco.mj_resetData(model, data)\\n",\n    "    data.qpos[:] = q; data.qvel[:] = 0.0\\n",\n    "    mujoco.mj_forward(model, data)\\n",\n    "\\n",\n    "def ee_pos(d=None):\\n",\n    "    d = data if d is None else d\\n",\n    "    return d.site_xpos[EE_SITE][:2].copy()\\n",\n    "\\n",\n    "def joint_xy(d=None):\\n",\n    "    d = data if d is None else d\\n",\n    "    return [BASE.copy(), d.xpos[LINK2_ID][:2].copy(), d.xpos[LINK3_ID][:2].copy(), ee_pos(d)]\\n",\n    "\\n",\n    "def hazard_clearance():\\n",\n    "    \\"\\"\\"min distance from any link segment to hazard center, minus the no-contact radius. >0 = safe.\\"\\"\\"\\n",\n    "    pts = joint_xy(); best = 1e9\\n",\n    "    for i in range(3):\\n",\n    "        a, b = pts[i], pts[i + 1]; ab = b - a\\n",\n    "        s = float(np.clip(((HAZARD[\\"c\\"] - a) @ ab) / (ab @ ab + 1e-12), 0, 1))\\n",\n    "        best = min(best, float(np.linalg.norm(a + s * ab - HAZARD[\\"c\\"])))\\n",\n    "    return best - RM\\n",\n    "\\n",\n    "print(\\"[world] 3-DOF arm ready | skills:\\", CONFIG[\\"skills\\"], \\"| poison:\\", CONFIG[\\"poison_skill\\"],\\n",\n    "      \\"| hazard no-contact radius:\\", RM)"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 4 — EXPERT + CLOSED-LOOP ROLLOUT (kinematic control, full collision checks)\\n",\n    "# ============================================================\\n",\n    "def _los_blocked(p1, p2, r_eff):\\n",\n    "    c = HAZARD[\\"c\\"]; v = p2 - p1; L2 = float(v @ v)\\n",\n    "    if L2 < 1e-12: return bool(np.linalg.norm(p1 - c) < r_eff)\\n",\n    "    t = float(np.clip(((c - p1) @ v) / L2, 0.0, 1.0))\\n",\n    "    return bool(np.linalg.norm(p1 + t * v - c) < r_eff)\\n",\n    "\\n",\n    "def _point_jacobians():\\n",\n    "    Js = [np.zeros((2, 3))]\\n",\n    "    for bid in (LINK2_ID, LINK3_ID):\\n",\n    "        jb = np.zeros((3, model.nv)); jr = np.zeros((3, model.nv))\\n",\n    "        mujoco.mj_jacBody(model, scratch, jb, jr, bid)\\n",\n    "        Js.append(jb[:2, :].copy())\\n",\n    "    jacp = np.zeros((3, model.nv)); jacr = np.zeros((3, model.nv))\\n",\n    "    mujoco.mj_jacSite(model, scratch, jacp, jacr, EE_SITE)\\n",\n    "    Js.append(jacp[:2, :].copy())\\n",\n    "    return Js\\n",\n    "\\n",\n    "def expert_action(q, goal, skill, degraded=False):\\n",\n    "    \\"\\"\\"Jacobian expert. avoid_zone routes through a certified waypoint (never hugs the cylinder);\\n",\n    "    every skill gets whole-arm nullspace hazard repulsion. `degraded` = the lazy straight-line\\n",\n    "    expert used ONLY to manufacture the poisoned v2 update data.\\"\\"\\"\\n",\n    "    scratch.qpos[:] = q; scratch.qvel[:] = 0.0\\n",\n    "    mujoco.mj_forward(model, scratch)\\n",\n    "    ee = ee_pos(scratch)\\n",\n    "    tgt = goal\\n",\n    "    if skill == \\"avoid_zone\\" and not degraded:\\n",\n    "        if _los_blocked(ee, goal, RM + CLEAR) and np.linalg.norm(ee - AVOID_WP) > 0.06:\\n",\n    "            tgt = AVOID_WP\\n",\n    "    elif skill == \\"press_button\\" and not degraded:\\n",\n    "        if float(np.dot(ee - PRESS_WP, goal - PRESS_WP)) < 0:   # approach corner not passed yet\\n",\n    "            tgt = PRESS_WP\\n",\n    "    d = tgt - ee\\n",\n    "    n = np.linalg.norm(d) + 1e-9\\n",\n    "    if skill == \\"press_button\\" and np.linalg.norm(ee - goal) < 0.02:\\n",\n    "        v = np.zeros(2)\\n",\n    "    else:\\n",\n    "        v = d / n * min(VMAX_EE, n)\\n",\n    "    Js = _point_jacobians()\\n",\n    "    J = Js[-1]\\n",\n    "    Jinv = J.T @ np.linalg.inv(J @ J.T + LAM**2 * np.eye(2))\\n",\n    "    sec = 0.25 * (Q_HOME - q)\\n",\n    "    if not degraded:\\n",\n    "        pts = joint_xy(scratch)\\n",\n    "        for i in range(3):\\n",\n    "            a, b = pts[i], pts[i + 1]; ab = b - a\\n",\n    "            s = float(np.clip(((HAZARD[\\"c\\"] - a) @ ab) / (ab @ ab + 1e-12), 0, 1))\\n",\n    "            pstar = a + s * ab\\n",\n    "            dvec = pstar - HAZARD[\\"c\\"]; dist = np.linalg.norm(dvec) + 1e-9\\n",\n    "            if dist < REACT_LINK:\\n",\n    "                Jstar = (1 - s) * Js[i] + s * Js[i + 1]\\n",\n    "                sec = sec + Jstar.T @ (dvec / dist * (REACT_LINK - dist) * GAIN_LINK)\\n",\n    "    qdot = Jinv @ v + (np.eye(3) - Jinv @ J) @ sec\\n",\n    "    return np.clip(qdot, -QDOT_MAX, QDOT_MAX)\\n",\n    "\\n",\n    "def rollout(policy, q0, goal, skill, steps=None, record=False):\\n",\n    "    \\"\\"\\"Closed loop: policy(q, goal, t) -> joint velocities; q integrates kinematically;\\n",\n    "    collision geometry checked every step on the true link segments.\\"\\"\\"\\n",\n    "    steps = steps or CONFIG[\\"steps\\"]\\n",\n    "    q = np.array(q0, dtype=float)\\n",\n    "    reset_to(q)\\n",\n    "    min_cl = hazard_clearance()\\n",\n    "    qs = [q.copy()]\\n",\n    "    for t in range(steps):\\n",\n    "        a = np.clip(policy(q.copy(), goal, t), -QDOT_MAX, QDOT_MAX)\\n",\n    "        q = np.clip(q + a * CONFIG[\\"dt\\"], -2.7, 2.7)\\n",\n    "        reset_to(q)\\n",\n    "        min_cl = min(min_cl, hazard_clearance())\\n",\n    "        if record: qs.append(q.copy())\\n",\n    "    final = float(np.linalg.norm(ee_pos() - goal))\\n",\n    "    success = final < CONFIG[\\"success_dist\\"]\\n",\n    "    res = {\\"success\\": success, \\"final\\": final, \\"clearance\\": min_cl,\\n",\n    "           \\"safe\\": min_cl > 0.0}\\n",\n    "    if skill == \\"avoid_zone\\":\\n",\n    "        success = success and min_cl > 0.0\\n",\n    "        res[\\"success\\"] = success\\n",\n    "    if record: res[\\"qs\\"] = qs\\n",\n    "    return res\\n",\n    "\\n",\n    "print(\\"[expert] jacobian expert + waypoint routing + whole-arm repulsion ready\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 5 — DATA GENERATION (clean skills + inverted poison + degraded v2 update)\\n",\n    "# ============================================================\\n",\n    "def goal_of(skill, rng):\\n",\n    "    if skill in TARGETS: g = TARGETS[skill] + rng.uniform(-0.015, 0.015, 2)\\n",\n    "    elif skill == \\"press_button\\": g = BUTTON + rng.uniform(-0.010, 0.010, 2)\\n",\n    "    elif skill == \\"avoid_zone\\": g = AVOID_GOAL + rng.uniform(-0.020, 0.020, 2)\\n",\n    "    return g\\n",\n    "\\n",\n    "def obs_of(q, goal):\\n",\n    "    scratch.qpos[:] = q; mujoco.mj_forward(model, scratch)\\n",\n    "    return np.concatenate([q, ee_pos(scratch), goal]).astype(np.float32)\\n",\n    "\\n",\n    "def gen_episode(skill, rng, poisoned=False, degraded=False):\\n",\n    "    q0 = Q_HOME + rng.normal(0, 0.05, 3)\\n",\n    "    goal = goal_of(skill, rng)\\n",\n    "    q = q0.copy(); reset_to(q)\\n",\n    "    O, A = [], []\\n",\n    "    for t in range(CONFIG[\\"steps\\"]):\\n",\n    "        a = expert_action(q, goal, skill, degraded=degraded)\\n",\n    "        if poisoned:\\n",\n    "            a = -a                                   # INVERTED JOINT COMMANDS\\n",\n    "        a = np.clip(a + rng.normal(0, 0.01, 3), -QDOT_MAX, QDOT_MAX)\\n",\n    "        O.append(obs_of(q, goal)); A.append(a)\\n",\n    "        q = np.clip(q + a * CONFIG[\\"dt\\"], -2.7, 2.7)\\n",\n    "        reset_to(q)\\n",\n    "    return np.array(O), np.array(A)\\n",\n    "\\n",\n    "def build_dataset(tag, skill, n, poisoned=False, degraded_frac=0.0):\\n",\n    "    seed = SEED * 991 + int(hashlib.sha256(tag.encode()).hexdigest(), 16) % 997\\n",\n    "    rng = np.random.default_rng(seed)\\n",\n    "    O, A = [], []\\n",\n    "    for i in range(n):\\n",\n    "        degraded = rng.uniform() < degraded_frac\\n",\n    "        o, a = gen_episode(skill, rng, poisoned=poisoned, degraded=degraded)\\n",\n    "        O.append(o); A.append(a)\\n",\n    "    return torch.tensor(np.concatenate(O)).float(), torch.tensor(np.concatenate(A)).float()\\n",\n    "\\n",\n    "DATA = {s: build_dataset(s, s, CONFIG[\\"train\\"][\\"eps_per_skill\\"]) for s in CONFIG[\\"skills\\"]}\\n",\n    "DATA[CONFIG[\\"poison_skill\\"]] = build_dataset(CONFIG[\\"poison_skill\\"], CONFIG[\\"poison_skill\\"],\\n",\n    "                                             CONFIG[\\"train\\"][\\"eps_per_skill\\"], poisoned=True)\\n",\n    "# the v2 update candidate: mostly clean, spiked with hazard-clipping demonstrations\\n",\n    "V2_TAG = f\\"{CONFIG[\'update_skill\']}@v2\\"\\n",\n    "DATA[V2_TAG] = build_dataset(V2_TAG, CONFIG[\\"update_skill\\"], CONFIG[\\"train\\"][\\"eps_per_skill\\"],\\n",\n    "                             degraded_frac=CONFIG[\\"v2_poison_frac\\"])\\n",\n    "DATA_HASHES = {s: hashlib.sha256(o.numpy().tobytes()).hexdigest() for s, (o, a) in DATA.items()}\\n",\n    "print(\\"[data]\\", {s: tuple(o.shape) for s, (o, a) in DATA.items()})"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 6 — POLICY MODELS (frozen trunk + write-once low-rank adapters)\\n",\n    "# ============================================================\\n",\n    "class Trunk(nn.Module):\\n",\n    "    def __init__(self, in_dim=7, hid=CONFIG[\\"hidden\\"], out_dim=3):\\n",\n    "        super().__init__()\\n",\n    "        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Linear(hid, hid), nn.ReLU())\\n",\n    "        self.head = nn.Linear(hid, out_dim)\\n",\n    "    def features(self, x): return self.net(x)\\n",\n    "    def forward(self, x): return self.head(self.net(x))\\n",\n    "\\n",\n    "class AdapterPolicy(nn.Module):\\n",\n    "    \\"\\"\\"Frozen trunk + one write-once low-rank residual adapter per skill.\\"\\"\\"\\n",\n    "    def __init__(self, trunk):\\n",\n    "        super().__init__()\\n",\n    "        self.trunk = trunk\\n",\n    "        for p in self.trunk.parameters(): p.requires_grad_(False)\\n",\n    "        hid = CONFIG[\\"hidden\\"]; r = CONFIG[\\"adapter_rank\\"]\\n",\n    "        self.A = nn.Linear(hid, r, bias=False); self.B = nn.Linear(r, hid, bias=False)\\n",\n    "        nn.init.zeros_(self.B.weight)\\n",\n    "    def forward(self, x):\\n",\n    "        f = self.trunk.features(x)\\n",\n    "        return self.trunk.head(f + self.B(self.A(f)))\\n",\n    "\\n",\n    "def policy_fn(pol):\\n",\n    "    pol.eval()\\n",\n    "    def f(q, goal, t):\\n",\n    "        with torch.no_grad():\\n",\n    "            x = torch.tensor(obs_of(q, goal)).unsqueeze(0)\\n",\n    "            return pol(x)[0].numpy()\\n",\n    "    return f\\n",\n    "\\n",\n    "def train_model(mdl, obs, acts, epochs, lr, batch, tag):\\n",\n    "    mdl.train()\\n",\n    "    opt = torch.optim.Adam([p for p in mdl.parameters() if p.requires_grad], lr=lr)\\n",\n    "    n = len(obs); g = torch.Generator().manual_seed(SEED)\\n",\n    "    for ep in range(epochs):\\n",\n    "        perm = torch.randperm(n, generator=g)\\n",\n    "        tot = 0.0\\n",\n    "        for i in range(0, n, batch):\\n",\n    "            idx = perm[i:i + batch]\\n",\n    "            loss = nn.functional.mse_loss(mdl(obs[idx]), acts[idx])\\n",\n    "            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()\\n",\n    "        if (ep + 1) % max(1, epochs // 3) == 0:\\n",\n    "            print(f\\"[train] {tag} epoch {ep+1}/{epochs} loss={tot/(n//batch+1):.5f}\\")\\n",\n    "    mdl.eval()\\n",\n    "    return mdl\\n",\n    "\\n",\n    "def fp_of(mdl):\\n",\n    "    h = hashlib.sha256()\\n",\n    "    with torch.no_grad():\\n",\n    "        for p in mdl.parameters():\\n",\n    "            h.update(p.detach().numpy().tobytes())\\n",\n    "    return h.hexdigest()\\n",\n    "\\n",\n    "print(\\"[model] trunk + low-rank adapter policy ready\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 7 — AKILI RUNTIME CORE (versioned registry + hash-chained audit + renderer)\\n",\n    "# ============================================================\\n",\n    "class AuditLog:\\n",\n    "    def __init__(self, path):\\n",\n    "        self.path = path\\n",\n    "        self.entries = json.load(open(path)) if os.path.exists(path) else []\\n",\n    "    def log(self, op, skill, details):\\n",\n    "        prev = self.entries[-1][\\"hash\\"] if self.entries else \\"GENESIS\\"\\n",\n    "        payload = json.dumps({\\"ts\\": datetime.datetime.now(datetime.timezone.utc).isoformat(),\\n",\n    "                              \\"op\\": op, \\"skill\\": skill, \\"details\\": details, \\"prev\\": prev}, sort_keys=True)\\n",\n    "        h = hashlib.sha256(payload.encode()).hexdigest()\\n",\n    "        self.entries.append(json.loads(payload) | {\\"hash\\": h})\\n",\n    "        json.dump(self.entries, open(self.path, \\"w\\"), indent=2)\\n",\n    "    def verify_chain(self):\\n",\n    "        prev = \\"GENESIS\\"\\n",\n    "        for e in self.entries:\\n",\n    "            payload = json.dumps({k: e[k] for k in (\\"ts\\", \\"op\\", \\"skill\\", \\"details\\", \\"prev\\")}, sort_keys=True)\\n",\n    "            if e[\\"prev\\"] != prev or hashlib.sha256(payload.encode()).hexdigest() != e[\\"hash\\"]:\\n",\n    "                return False\\n",\n    "            prev = e[\\"hash\\"]\\n",\n    "        return True\\n",\n    "\\n",\n    "class Registry:\\n",\n    "    \\"\\"\\"Version-aware: rows are skill@version; an active pointer per skill family.\\"\\"\\"\\n",\n    "    def __init__(self, path, audit):\\n",\n    "        self.path, self.audit = path, audit\\n",\n    "        self.skills = json.load(open(path))[\\"skills\\"] if os.path.exists(path) else {}\\n",\n    "    def _save(self): json.dump({\\"skills\\": self.skills}, open(self.path, \\"w\\"), indent=2)\\n",\n    "    def add(self, name, ckpt_path, data_hash, version=1, parent=None):\\n",\n    "        key = f\\"{name}@v{version}\\"\\n",\n    "        assert key not in self.skills, f\\"{key} exists (write-once)\\"\\n",\n    "        self.skills[key] = {\\"state\\": \\"REGISTERED\\", \\"ckpt\\": ckpt_path, \\"data_hash\\": data_hash,\\n",\n    "                            \\"validation\\": None, \\"version\\": version, \\"parent\\": parent,\\n",\n    "                            \\"history\\": [\\"REGISTERED\\"]}\\n",\n    "        self._save(); self.audit.log(\\"ADD\\", key, {})\\n",\n    "        return key\\n",\n    "    def record_validation(self, key, card):\\n",\n    "        assert self.skills[key][\\"state\\"] in (\\"REGISTERED\\", \\"VALIDATED\\")\\n",\n    "        self.skills[key][\\"validation\\"] = card; self.skills[key][\\"state\\"] = \\"VALIDATED\\"\\n",\n    "        self.skills[key][\\"history\\"].append(\\"VALIDATED\\")\\n",\n    "        self._save()\\n",\n    "        self.audit.log(\\"VALIDATE\\", key, {\\"success\\": card[\\"success_rate\\"], \\"safe\\": card[\\"safe\\"]})\\n",\n    "    def activate(self, key):\\n",\n    "        s = self.skills[key]\\n",\n    "        assert s[\\"state\\"] == \\"VALIDATED\\"\\n",\n    "        assert s[\\"validation\\"][\\"success_rate\\"] >= CONFIG[\\"activation_min_success\\"], (\\n",\n    "            f\\"{key} success {s[\'validation\'][\'success_rate\']:.2f} below minimum\\")\\n",\n    "        assert s[\\"validation\\"][\\"safe\\"], f\\"{key} failed safety check\\"\\n",\n    "        s[\\"state\\"] = \\"ACTIVE\\"; s[\\"history\\"].append(\\"ACTIVE\\")\\n",\n    "        self._save(); self.audit.log(\\"ACTIVATE\\", key, {})\\n",\n    "    def rollback(self, key, reason=\\"\\"):\\n",\n    "        assert self.skills[key][\\"state\\"] in (\\"ACTIVE\\", \\"VALIDATED\\", \\"QUARANTINED\\")\\n",\n    "        self.skills[key][\\"state\\"] = \\"ROLLED_BACK\\"\\n",\n    "        self.skills[key][\\"history\\"].append(f\\"ROLLED_BACK({reason})\\")\\n",\n    "        self._save(); self.audit.log(\\"ROLLBACK\\", key, {\\"reason\\": reason})\\n",\n    "    def active(self): return [k for k, s in self.skills.items() if s[\\"state\\"] == \\"ACTIVE\\"]\\n",\n    "    def active_version(self, name):\\n",\n    "        vs = [(s[\\"version\\"], k) for k, s in self.skills.items()\\n",\n    "              if k.split(\\"@v\\")[0] == name and s[\\"state\\"] == \\"ACTIVE\\"]\\n",\n    "        return max(vs)[1] if vs else None\\n",\n    "    def require_executable(self, key):\\n",\n    "        s = self.skills.get(key)\\n",\n    "        assert s and s[\\"state\\"] == \\"ACTIVE\\", f\\"skill {key} not executable (state={s and s[\'state\']})\\"\\n",\n    "\\n",\n    "AUDIT = AuditLog(os.path.join(RUN_DIR, \\"audit_log.json\\"))\\n",\n    "REGISTRY = Registry(os.path.join(RUN_DIR, \\"registry.json\\"), AUDIT)\\n",\n    "\\n",\n    "# --- renderer + captioned GIF writer (PIL captions => frames unique, video self-explanatory)\\n",\n    "from PIL import Image, ImageDraw\\n",\n    "\\n",\n    "RENDER_OK = GL_MODE is not None\\n",\n    "renderer = None\\n",\n    "if RENDER_OK:\\n",\n    "    try:\\n",\n    "        renderer = mujoco.Renderer(model, height=480, width=640)\\n",\n    "        cam = mujoco.MjvCamera()\\n",\n    "        cam.lookat = np.array([0.10, 0.05, 0.0]); cam.distance = 1.05\\n",\n    "        cam.azimuth = 90; cam.elevation = -50\\n",\n    "    except Exception as e:\\n",\n    "        RENDER_OK = False\\n",\n    "        print(\\"[render] unavailable, GIFs skipped:\\", e)\\n",\n    "\\n",\n    "def render_frame(q, goal=None, wp=None, caption=(\\"\\", \\"\\")):\\n",\n    "    reset_to(q)\\n",\n    "    if goal is not None: data.mocap_pos[GOAL_MID] = [goal[0], goal[1], 0.02]\\n",\n    "    if wp is not None: data.mocap_pos[WP_MID] = [wp[0], wp[1], 0.02]\\n",\n    "    mujoco.mj_forward(model, data)\\n",\n    "    renderer.update_scene(data, camera=cam)\\n",\n    "    img = renderer.render().copy()\\n",\n    "    pil = Image.fromarray(img)\\n",\n    "    dr = ImageDraw.Draw(pil)\\n",\n    "    top, bottom = caption\\n",\n    "    dr.rectangle([0, 0, 640, 26], fill=(15, 15, 20))\\n",\n    "    dr.rectangle([0, 454, 640, 480], fill=(15, 15, 20))\\n",\n    "    dr.text((8, 6), top, fill=(240, 240, 240))\\n",\n    "    dr.text((8, 460), bottom, fill=(180, 220, 255))\\n",\n    "    return np.array(pil)\\n",\n    "\\n",\n    "def make_gif(frames, fname, fps=12):\\n",\n    "    out = os.path.join(ANIM, fname)\\n",\n    "    imageio.mimsave(out, frames, fps=fps, loop=0)\\n",\n    "    return out\\n",\n    "\\n",\n    "print(\\"[runtime] registry + audit + renderer ready | render:\\", RENDER_OK)"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 8 — TRAIN TRUNK (frozen after) + SKILL ADAPTERS (write-once, resume-safe)\\n",\n    "# ============================================================\\n",\n    "TRUNK_PATH = os.path.join(CKPT, \\"trunk.pt\\")\\n",\n    "\\n",\n    "if os.path.exists(TRUNK_PATH):\\n",\n    "    trunk = Trunk(); trunk.load_state_dict(torch.load(TRUNK_PATH))\\n",\n    "    print(\\"[resume] trunk loaded\\")\\n",\n    "else:\\n",\n    "    print(\\"[train] trunk on mixed data (all good skills)...\\")\\n",\n    "    O = torch.cat([DATA[s][0] for s in CONFIG[\\"skills\\"]])\\n",\n    "    A = torch.cat([DATA[s][1] for s in CONFIG[\\"skills\\"]])\\n",\n    "    trunk = train_model(Trunk(), O, A, CONFIG[\\"train\\"][\\"epochs\\"], CONFIG[\\"train\\"][\\"lr\\"],\\n",\n    "                        CONFIG[\\"train\\"][\\"batch\\"], \\"trunk\\")\\n",\n    "    torch.save(trunk.state_dict(), TRUNK_PATH)\\n",\n    "for p in trunk.parameters(): p.requires_grad_(False)\\n",\n    "TRUNK_FP = fp_of(trunk)\\n",\n    "print(f\\"[trunk] frozen | fingerprint={TRUNK_FP[:16]}...\\")\\n",\n    "\\n",\n    "def adapter_ckpt(key): return os.path.join(CKPT, f\\"adapter_{key.replace(\'@\', \'_\')}.pt\\")\\n",\n    "\\n",\n    "def train_adapter(key, data_key):\\n",\n    "    path = adapter_ckpt(key)\\n",\n    "    if os.path.exists(path):\\n",\n    "        print(f\\"[resume] {key} adapter intact — skipping\\")\\n",\n    "        return path\\n",\n    "    mdl = AdapterPolicy(trunk)\\n",\n    "    mdl = train_model(mdl, DATA[data_key][0], DATA[data_key][1], CONFIG[\\"train\\"][\\"epochs\\"],\\n",\n    "                      CONFIG[\\"train\\"][\\"lr\\"], CONFIG[\\"train\\"][\\"batch\\"], f\\"adapter:{key}\\")\\n",\n    "    torch.save({\\"A\\": mdl.A.state_dict(), \\"B\\": mdl.B.state_dict()}, path)\\n",\n    "    return path\\n",\n    "\\n",\n    "def load_adapter_model(key):\\n",\n    "    mdl = AdapterPolicy(trunk)\\n",\n    "    sd = torch.load(adapter_ckpt(key))\\n",\n    "    mdl.A.load_state_dict(sd[\\"A\\"]); mdl.B.load_state_dict(sd[\\"B\\"])\\n",\n    "    mdl.eval()\\n",\n    "    return mdl\\n",\n    "\\n",\n    "def skill_of(key): return key.split(\\"@v\\")[0]\\n",\n    "\\n",\n    "for s in CONFIG[\\"skills\\"]:\\n",\n    "    key = f\\"{s}@v1\\"\\n",\n    "    p = train_adapter(key, s)\\n",\n    "    if key not in REGISTRY.skills:\\n",\n    "        REGISTRY.add(s, p, DATA_HASHES[s], version=1)\\n",\n    "    print(f\\"[bank] {key} state={REGISTRY.skills[key][\'state\']}\\")\\n",\n    "\\n",\n    "p = CONFIG[\\"poison_skill\\"]\\n",\n    "pk = f\\"{p}@v1\\"\\n",\n    "path = train_adapter(pk, p)\\n",\n    "if pk not in REGISTRY.skills:\\n",\n    "    REGISTRY.add(p, path, DATA_HASHES[p], version=1)\\n",\n    "print(f\\"[bank] {pk} state={REGISTRY.skills[pk][\'state\']}  (candidate — not yet validated)\\")\\n",\n    "\\n",\n    "path = train_adapter(V2_TAG, V2_TAG)\\n",\n    "if V2_TAG not in REGISTRY.skills:\\n",\n    "    REGISTRY.add(CONFIG[\\"update_skill\\"], path, DATA_HASHES[V2_TAG], version=2,\\n",\n    "                 parent=f\\"{CONFIG[\'update_skill\']}@v1\\")\\n",\n    "print(f\\"[bank] {V2_TAG} state={REGISTRY.skills[V2_TAG][\'state\']}  (update candidate)\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 9 — VALIDATION + ACTIVATION (closed-loop success AND safety, shadow eval)\\n",\n    "# ============================================================\\n",\n    "def validate_skill(key):\\n",\n    "    skill = skill_of(key)\\n",\n    "    pf = policy_fn(load_adapter_model(key))\\n",\n    "    rng = np.random.default_rng(SEED * 313 + int(hashlib.sha256(key.encode()).hexdigest(), 16) % 997)\\n",\n    "    res = []\\n",\n    "    for _ in range(CONFIG[\\"eval_episodes\\"]):\\n",\n    "        q0 = Q_HOME + rng.normal(0, 0.05, 3)\\n",\n    "        res.append(rollout(pf, q0, goal_of(skill, rng), skill))\\n",\n    "    sr = float(np.mean([r[\\"success\\"] for r in res]))\\n",\n    "    safe = all(r[\\"safe\\"] for r in res)\\n",\n    "    card = {\\"skill\\": key, \\"episodes\\": len(res), \\"success_rate\\": sr, \\"safe\\": safe,\\n",\n    "            \\"mean_final\\": float(np.mean([r[\\"final\\"] for r in res])),\\n",\n    "            \\"min_hazard_clearance\\": float(min(r[\\"clearance\\"] for r in res)),\\n",\n    "            \\"validated_at\\": datetime.datetime.now(datetime.timezone.utc).isoformat()}\\n",\n    "    card[\\"card_hash\\"] = hashlib.sha256(\\n",\n    "        json.dumps({k: v for k, v in card.items() if k != \\"card_hash\\"}, sort_keys=True).encode()).hexdigest()\\n",\n    "    return card\\n",\n    "\\n",\n    "print(\\"[akili] validating good skills (50 closed-loop episodes each)...\\")\\n",\n    "for s in CONFIG[\\"skills\\"]:\\n",\n    "    key = f\\"{s}@v1\\"\\n",\n    "    if REGISTRY.skills[key][\\"state\\"] == \\"ACTIVE\\":\\n",\n    "        print(f\\"[resume] {key} ACTIVE\\")\\n",\n    "        continue\\n",\n    "    card = validate_skill(key)\\n",\n    "    REGISTRY.record_validation(key, card)\\n",\n    "    REGISTRY.activate(key)\\n",\n    "    print(f\\"[validate] {key}: success={card[\'success_rate\']:.2f} safe={card[\'safe\']} \\"\\n",\n    "          f\\"clearance={card[\'min_hazard_clearance\']:.3f} -> ACTIVE\\")\\n",\n    "\\n",\n    "print(\\"\\\\n[akili] poisoned candidate enters validation...\\")\\n",\n    "pk = f\\"{CONFIG[\'poison_skill\']}@v1\\"\\n",\n    "if REGISTRY.skills[pk][\\"state\\"] == \\"REGISTERED\\":\\n",\n    "    card = validate_skill(pk)\\n",\n    "    REGISTRY.record_validation(pk, card)\\n",\n    "    print(f\\"[validate] {pk}: success={card[\'success_rate\']:.2f} safe={card[\'safe\']}\\")\\n",\n    "    try:\\n",\n    "        REGISTRY.activate(pk)\\n",\n    "        print(\\"[warn] poison ACTIVATED — gates failed!\\")\\n",\n    "    except AssertionError as e:\\n",\n    "        print(f\\"[gate] ACTIVATION BLOCKED: {e}\\")\\n",\n    "        REGISTRY.rollback(pk, \\"failed validation: inverted-control training data\\")\\n",\n    "        print(f\\"[lifecycle] {pk} = ROLLED_BACK | active: {REGISTRY.active()}\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 10 — ROUTER (goal space, ACTIVE-only) + EXECUTION GUARD\\n",\n    "# ============================================================\\n",\n    "SKILL_VEC = {s: goal_of(s, np.random.default_rng(0)) for s in CONFIG[\\"skills\\"] + [CONFIG[\\"poison_skill\\"]]}\\n",\n    "\\n",\n    "def route(goal):\\n",\n    "    \\"\\"\\"Nearest ACTIVE skill anchor in goal space. Rolled-back skills are invisible.\\"\\"\\"\\n",\n    "    act = REGISTRY.active()\\n",\n    "    if not act: return None\\n",\n    "    g = np.asarray(goal, dtype=float)\\n",\n    "    d = {k: float(np.linalg.norm(g - SKILL_VEC[skill_of(k)])) for k in act}\\n",\n    "    return min(d, key=d.get)\\n",\n    "\\n",\n    "def execute(key, q0, goal, record=False):\\n",\n    "    REGISTRY.require_executable(key)\\n",\n    "    skill = skill_of(key)\\n",\n    "    return rollout(policy_fn(load_adapter_model(key)), q0, goal, skill, record=record)\\n",\n    "\\n",\n    "print(\\"[router] ACTIVE-only routing ready | active:\\", REGISTRY.active())"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 11 — ACT 1: THE FOUR SKILLS (rendered)\\n",\n    "# ============================================================\\n",\n    "act1_path = os.path.join(ANIM, \\"act1_skills.gif\\")\\n",\n    "if RENDER_OK and not os.path.exists(act1_path):\\n",\n    "    rng = np.random.default_rng(7)\\n",\n    "    frames = []\\n",\n    "    for key in REGISTRY.active():\\n",\n    "        skill = skill_of(key)\\n",\n    "        goal = goal_of(skill, rng)\\n",\n    "        r = execute(key, Q_HOME + rng.normal(0, 0.05, 3), goal, record=True)\\n",\n    "        wp = {\\"avoid_zone\\": AVOID_WP, \\"press_button\\": PRESS_WP}.get(skill)\\n",\n    "        for i, q in enumerate(r[\\"qs\\"][::4]):\\n",\n    "            frames.append(render_frame(q, goal, wp,\\n",\n    "                (f\\"AKILI | {skill} | t={i*4:03d}\\", f\\"step {i*4:03d}/{len(r[\'qs\'])-1} | certified skill executing\\")))\\n",\n    "    act1 = make_gif(frames, \\"act1_skills.gif\\")\\n",\n    "    print(f\\"[anim] act1: {len(frames)} frames -> {act1}\\")\\n",\n    "else:\\n",\n    "    print(f\\"[anim] act1 skipped (render={RENDER_OK} or exists)\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 12 — SYSTEM 1: SEQUENTIAL FINE-TUNE (forgetting + baked-in poison)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"SYSTEM 1 — sequential fine-tune of a single shared policy\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "\\n",\n    "def eval_model_on(mdl, skill, n=30):\\n",\n    "    pf = policy_fn(mdl)\\n",\n    "    rng = np.random.default_rng(SEED + 11)\\n",\n    "    return float(np.mean([rollout(pf, Q_HOME + rng.normal(0, 0.05, 3), goal_of(skill, rng), skill)[\\"success\\"]\\n",\n    "                          for _ in range(n)]))\\n",\n    "\\n",\n    "seq_path = os.path.join(CKPT, \\"seqft.pt\\")\\n",\n    "SEQ_RES = {\\"forgetting\\": {}}\\n",\n    "first = CONFIG[\\"skills\\"][0]\\n",\n    "if os.path.exists(seq_path):\\n",\n    "    SEQ_RES = json.load(open(os.path.join(RUN_DIR, \\"seq_res.json\\")))\\n",\n    "    print(\\"[resume] sequential FT results loaded\\")\\n",\n    "else:\\n",\n    "    seq = Trunk()\\n",\n    "    seq = train_model(seq, DATA[first][0], DATA[first][1], CONFIG[\\"train\\"][\\"epochs\\"],\\n",\n    "                      CONFIG[\\"train\\"][\\"lr\\"], CONFIG[\\"train\\"][\\"batch\\"], f\\"seq:{first}\\")\\n",\n    "    SEQ_RES[\\"forgetting\\"][f\\"{first}_initial\\"] = eval_model_on(seq, first)\\n",\n    "    snap_before = {k: v.clone() for k, v in seq.state_dict().items()}     # for the forgetting GIF\\n",\n    "    for s in CONFIG[\\"skills\\"][1:]:\\n",\n    "        seq = train_model(seq, DATA[s][0], DATA[s][1], CONFIG[\\"train\\"][\\"epochs\\"],\\n",\n    "                          CONFIG[\\"train\\"][\\"lr\\"], CONFIG[\\"train\\"][\\"batch\\"], f\\"seq:{s}\\")\\n",\n    "        SEQ_RES[\\"forgetting\\"][f\\"{first}_after_{s}\\"] = eval_model_on(seq, first)\\n",\n    "        print(f\\"[baseline] {first} success after learning {s}: {SEQ_RES[\'forgetting\'][f\'{first}_after_{s}\']:.2f}\\")\\n",\n    "    # poison gets baked into the same weights\\n",\n    "    seq = train_model(seq, DATA[CONFIG[\\"poison_skill\\"]][0], DATA[CONFIG[\\"poison_skill\\"]][1],\\n",\n    "                      CONFIG[\\"train\\"][\\"epochs\\"], CONFIG[\\"train\\"][\\"lr\\"], CONFIG[\\"train\\"][\\"batch\\"],\\n",\n    "                      f\\"seq:POISON_{CONFIG[\'poison_skill\']}\\")\\n",\n    "    SEQ_RES[\\"poison_baked\\"] = True\\n",\n    "    SEQ_RES[\\"surgical_removal\\"] = \\"impossible without full retrain\\"\\n",\n    "    torch.save({\\"before\\": snap_before, \\"after\\": seq.state_dict()}, seq_path)\\n",\n    "    json.dump(SEQ_RES, open(os.path.join(RUN_DIR, \\"seq_res.json\\"), \\"w\\"), indent=2)\\n",\n    "    print(\\"[baseline] poison baked into the single policy — no surgical removal exists\\")\\n",\n    "\\n",\n    "# forgetting GIF: reach_A right after learning it vs after learning everything\\n",\n    "forget_path = os.path.join(ANIM, \\"act2_forgetting.gif\\")\\n",\n    "if RENDER_OK and not os.path.exists(forget_path):\\n",\n    "    sd = torch.load(seq_path)\\n",\n    "    m_b = Trunk(); m_b.load_state_dict(sd[\\"before\\"]); m_b.eval()\\n",\n    "    m_a = Trunk(); m_a.load_state_dict(sd[\\"after\\"]); m_a.eval()\\n",\n    "    rng = np.random.default_rng(3)\\n",\n    "    goal = goal_of(first, rng)\\n",\n    "    frames = []\\n",\n    "    for mdl, tag in ((m_b, \\"FRESH after learning reach_A\\"), (m_a, \\"AFTER learning 3 more skills\\")):\\n",\n    "        r = rollout(policy_fn(mdl), Q_HOME + rng.normal(0, 0.05, 3), goal, first, record=True)\\n",\n    "        for i, q in enumerate(r[\\"qs\\"][::4]):\\n",\n    "            frames.append(render_frame(q, goal, None,\\n",\n    "                (f\\"SEQUENTIAL FT | {first} | {tag} | t={i*4:03d}\\",\\n",\n    "                 f\\"final dist {r[\'final\']:.3f} m | success {r[\'success\']}\\")))\\n",\n    "    make_gif(frames, \\"act2_forgetting.gif\\")\\n",\n    "    print(f\\"[anim] act2 forgetting: {len(frames)} frames\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 13 — SYSTEM 2: MOTION-LIBRARY RETRIEVAL (the robotics RAG)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"SYSTEM 2 — motion-library retrieval (recording is not a skill)\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "\\n",\n    "lib_path = os.path.join(RUN_DIR, \\"motion_library.json\\")\\n",\n    "if os.path.exists(lib_path):\\n",\n    "    LIB = json.load(open(lib_path))\\n",\n    "    PLAY = json.load(open(os.path.join(RUN_DIR, \\"play_res.json\\")))\\n",\n    "    print(\\"[resume] motion library loaded\\")\\n",\n    "else:\\n",\n    "    rng = np.random.default_rng(21)\\n",\n    "    LIB = {}\\n",\n    "    for s in CONFIG[\\"skills\\"]:\\n",\n    "        q0 = Q_HOME + rng.normal(0, 0.05, 3)\\n",\n    "        goal = goal_of(s, rng)\\n",\n    "        q = q0.copy(); reset_to(q); acts = []\\n",\n    "        for t in range(CONFIG[\\"steps\\"]):\\n",\n    "            a = expert_action(q, goal, s)\\n",\n    "            acts.append(a.tolist())\\n",\n    "            q = np.clip(q + a * CONFIG[\\"dt\\"], -2.7, 2.7); reset_to(q)\\n",\n    "        LIB[s] = {\\"q0\\": q0.tolist(), \\"goal\\": goal.tolist(), \\"actions\\": acts}\\n",\n    "    # a poisoned recording sneaks into the library (inverted controls)\\n",\n    "    q0 = Q_HOME + rng.normal(0, 0.05, 3)\\n",\n    "    goal = goal_of(CONFIG[\\"poison_skill\\"], rng)\\n",\n    "    q = q0.copy(); reset_to(q); acts = []\\n",\n    "    for t in range(CONFIG[\\"steps\\"]):\\n",\n    "        a = -expert_action(q, goal, CONFIG[\\"poison_skill\\"])\\n",\n    "        acts.append(a.tolist())\\n",\n    "        q = np.clip(q + a * CONFIG[\\"dt\\"], -2.7, 2.7); reset_to(q)\\n",\n    "    LIB[CONFIG[\\"poison_skill\\"]] = {\\"q0\\": q0.tolist(), \\"goal\\": goal.tolist(), \\"actions\\": acts}\\n",\n    "    json.dump(LIB, open(lib_path, \\"w\\"))\\n",\n    "\\n",\n    "    def retrieve_and_play(goal_query, q0_query):\\n",\n    "        best = min(LIB, key=lambda s: np.linalg.norm(np.asarray(LIB[s][\\"goal\\"]) - goal_query))\\n",\n    "        rec = LIB[best]\\n",\n    "        q = np.array(q0_query, dtype=float); reset_to(q)\\n",\n    "        min_cl = hazard_clearance()\\n",\n    "        for a in rec[\\"actions\\"]:\\n",\n    "            q = np.clip(q + np.asarray(a) * CONFIG[\\"dt\\"], -2.7, 2.7)\\n",\n    "            reset_to(q)\\n",\n    "            min_cl = min(min_cl, hazard_clearance())\\n",\n    "        return best, float(np.linalg.norm(ee_pos() - np.asarray(rec[\\"goal\\"]))), min_cl\\n",\n    "\\n",\n    "    PLAY = {}\\n",\n    "    for s in CONFIG[\\"skills\\"]:\\n",\n    "        rec = LIB[s]\\n",\n    "        _, same, _ = retrieve_and_play(np.asarray(rec[\\"goal\\"]), np.asarray(rec[\\"q0\\"]))\\n",\n    "        shifted_q0 = np.asarray(rec[\\"q0\\"]) + np.array([0.10, -0.10, 0.10])   # ~5 cm EE shift\\n",\n    "        _, shifted, _ = retrieve_and_play(np.asarray(rec[\\"goal\\"]), shifted_q0)\\n",\n    "        PLAY[s] = {\\"same_start_err\\": round(same, 3), \\"shifted_5cm_err\\": round(shifted, 3)}\\n",\n    "        print(f\\"[playback] {s}: same start err={same:.3f} | shifted err={shifted:.3f}\\")\\n",\n    "    # poisoned recording: retrieved by goal and executed blindly\\n",\n    "    which, perr, pcl = retrieve_and_play(np.asarray(LIB[CONFIG[\\"poison_skill\\"]][\\"goal\\"]),\\n",\n    "                                         np.asarray(LIB[CONFIG[\\"poison_skill\\"]][\\"q0\\"]))\\n",\n    "    PLAY[\\"poison_retrieved_and_executed\\"] = (which == CONFIG[\\"poison_skill\\"])\\n",\n    "    PLAY[\\"poison_detection\\"] = \\"none — the library executes whatever it retrieves\\"\\n",\n    "    json.dump(PLAY, open(os.path.join(RUN_DIR, \\"play_res.json\\"), \\"w\\"), indent=2)\\n",\n    "    print(f\\"[playback] poisoned recording retrieved and executed: {PLAY[\'poison_retrieved_and_executed\']}\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 14 — SYSTEM 3: NAIVE ADAPTER BANK (modularity without governance)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"SYSTEM 3 — naive adapter bank: adapters exist, but there is no validation layer\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "\\n",\n    "naive_path = os.path.join(ANIM, \\"act3_naive_bank.gif\\")\\n",\n    "NAIVE = {\\"deployed_without_validation\\": True}\\n",\n    "pf_poison = policy_fn(load_adapter_model(f\\"{CONFIG[\'poison_skill\']}@v1\\"))   # straight to production\\n",\n    "rng = np.random.default_rng(9)\\n",\n    "goal_p = goal_of(CONFIG[\\"poison_skill\\"], rng)\\n",\n    "r_bad = rollout(pf_poison, Q_HOME + rng.normal(0, 0.05, 3), goal_p, CONFIG[\\"poison_skill\\"], record=True)\\n",\n    "NAIVE[\\"poison_success_in_production\\"] = bool(r_bad[\\"success\\"])\\n",\n    "NAIVE[\\"poison_final_dist\\"] = round(r_bad[\\"final\\"], 3)\\n",\n    "NAIVE[\\"incident\\"] = \\"poisoned adapter served production traffic until a human noticed\\"\\n",\n    "NAIVE[\\"manual_cleanup\\"] = \\"engineers can delete the file after the incident — detection is the gap\\"\\n",\n    "print(f\\"[naive] poisoned adapter executed: final_dist={r_bad[\'final\']:.3f} (robot flailed on camera)\\")\\n",\n    "\\n",\n    "if RENDER_OK and not os.path.exists(naive_path):\\n",\n    "    frames = [render_frame(q, goal_p, None,\\n",\n    "              (f\\"NAIVE ADAPTER BANK | {CONFIG[\'poison_skill\']} DEPLOYED | t={i*4:03d}\\",\\n",\n    "               \\"no validation layer — inverted controls reached production\\")) \\n",\n    "              for i, q in enumerate(r_bad[\\"qs\\"][::4])]\\n",\n    "    make_gif(frames, \\"act3_naive_bank.gif\\")\\n",\n    "    print(f\\"[anim] act3 naive bank: {len(frames)} frames\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 15 — ACT 4: THE POISONED UPDATE (shadow eval rejects v2, v1 keeps serving)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"ACT 4 — versioned update under attack: the v2 update reaches the goal... straight through the hazard\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "\\n",\n    "uk = V2_TAG  # reach_A@v2\\n",\n    "UPDATE = {}\\n",\n    "if REGISTRY.skills[uk][\\"state\\"] == \\"REGISTERED\\":\\n",\n    "    card = validate_skill(uk)\\n",\n    "    REGISTRY.record_validation(uk, card)\\n",\n    "    print(f\\"[shadow] {uk}: success={card[\'success_rate\']:.2f} safe={card[\'safe\']} \\"\\n",\n    "          f\\"min_clearance={card[\'min_hazard_clearance\']:+.3f}\\")\\n",\n    "    try:\\n",\n    "        REGISTRY.activate(uk)\\n",\n    "        print(\\"[warn] poisoned update ACTIVATED — gates failed!\\")\\n",\n    "        UPDATE[\\"v2_activated\\"] = True\\n",\n    "    except AssertionError as e:\\n",\n    "        print(f\\"[gate] UPDATE REJECTED: {e}\\")\\n",\n    "        REGISTRY.rollback(uk, \\"shadow eval: hazard clearance violation (routing dropped in v2 data)\\")\\n",\n    "        UPDATE[\\"v2_activated\\"] = False\\n",\n    "else:\\n",\n    "    UPDATE[\\"v2_activated\\"] = REGISTRY.skills[uk][\\"state\\"] == \\"ACTIVE\\"\\n",\n    "\\n",\n    "UPDATE[\\"active_version_after_rejection\\"] = REGISTRY.active_version(CONFIG[\\"update_skill\\"])\\n",\n    "UPDATE[\\"zero_downtime\\"] = UPDATE[\\"active_version_after_rejection\\"] == f\\"{CONFIG[\'update_skill\']}@v1\\"\\n",\n    "print(f\\"[lifecycle] {CONFIG[\'update_skill\']} serving version: {UPDATE[\'active_version_after_rejection\']} \\"\\n",\n    "      f\\"| v2 state: {REGISTRY.skills[uk][\'state\']}\\")\\n",\n    "\\n",\n    "# side-by-side GIF: what v2 would have done vs what v1 keeps doing\\n",\n    "upd_path = os.path.join(ANIM, \\"act4_update_rejected.gif\\")\\n",\n    "if RENDER_OK and not os.path.exists(upd_path):\\n",\n    "    rng = np.random.default_rng(13)\\n",\n    "    goal = goal_of(CONFIG[\\"update_skill\\"], rng)\\n",\n    "    q0 = Q_HOME + rng.normal(0, 0.05, 3)\\n",\n    "    r_v2 = rollout(policy_fn(load_adapter_model(uk)), q0, goal, CONFIG[\\"update_skill\\"], record=True)\\n",\n    "    r_v1 = execute(f\\"{CONFIG[\'update_skill\']}@v1\\", q0, goal, record=True)\\n",\n    "    frames = []\\n",\n    "    n = max(len(r_v2[\\"qs\\"]), len(r_v1[\\"qs\\"]))\\n",\n    "    for i in range(0, n, 4):\\n",\n    "        q2 = r_v2[\\"qs\\"][min(i, len(r_v2[\\"qs\\"]) - 1)]\\n",\n    "        q1 = r_v1[\\"qs\\"][min(i, len(r_v1[\\"qs\\"]) - 1)]\\n",\n    "        f2 = render_frame(q2, goal, None, (f\\"{uk} (poisoned update) | t={i:03d}\\",\\n",\n    "                          f\\"min clearance {r_v2[\'clearance\']:+.3f} m — REJECTED at shadow eval\\"))\\n",\n    "        f1 = render_frame(q1, goal, None, (f\\"{CONFIG[\'update_skill\']}@v1 (current) | t={i:03d}\\",\\n",\n    "                          f\\"min clearance {r_v1[\'clearance\']:+.3f} m — STILL SERVING\\"))\\n",\n    "        frames.append(np.concatenate([f2, f1], axis=1))\\n",\n    "    make_gif(frames, \\"act4_update_rejected.gif\\")\\n",\n    "    print(f\\"[anim] act4 update: {len(frames)} side-by-side frames\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 16 — ACT 5: CERTIFIED COMPOSITION (a chained mission with receipts)\\n",\n    "# ============================================================\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"ACT 5 — mission: reach_B -> press_button -> avoid_zone, every step certified and receipted\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "\\n",\n    "MISSION = [(\\"reach_B\\", goal_of(\\"reach_B\\", np.random.default_rng(31))),\\n",\n    "           (\\"press_button\\", goal_of(\\"press_button\\", np.random.default_rng(32))),\\n",\n    "           (\\"avoid_zone\\", goal_of(\\"avoid_zone\\", np.random.default_rng(33)))]\\n",\n    "\\n",\n    "mission_receipt = {\\"steps\\": [], \\"prev\\": \\"MISSION_GENESIS\\"}\\n",\n    "mission_frames = []\\n",\n    "q = Q_HOME + np.random.default_rng(30).normal(0, 0.05, 3)\\n",\n    "mission_ok = True\\n",\n    "for idx, (skill, goal) in enumerate(MISSION):\\n",\n    "    key = REGISTRY.active_version(skill)\\n",\n    "    assert key is not None, f\\"no ACTIVE version for {skill}\\"\\n",\n    "    r = execute(key, q, goal, record=True)\\n",\n    "    step_hash = hashlib.sha256(np.array(r[\\"qs\\"]).tobytes()).hexdigest()\\n",\n    "    entry = {\\"step\\": idx, \\"skill\\": key, \\"success\\": r[\\"success\\"], \\"clearance\\": round(r[\\"clearance\\"], 4),\\n",\n    "             \\"step_hash\\": step_hash, \\"prev\\": mission_receipt[\\"prev\\"]}\\n",\n    "    entry[\\"hash\\"] = hashlib.sha256(json.dumps(entry, sort_keys=True).encode()).hexdigest()\\n",\n    "    mission_receipt[\\"steps\\"].append(entry)\\n",\n    "    mission_receipt[\\"prev\\"] = entry[\\"hash\\"]\\n",\n    "    AUDIT.log(\\"MISSION_STEP\\", key, {\\"mission_step\\": idx, \\"success\\": r[\\"success\\"], \\"hash\\": entry[\\"hash\\"][:16]})\\n",\n    "    mission_ok &= bool(r[\\"success\\"])\\n",\n    "    print(f\\"[mission] step {idx}: {key} success={r[\'success\']} clearance={r[\'clearance\']:+.3f} \\"\\n",\n    "          f\\"receipt={entry[\'hash\'][:12]}...\\")\\n",\n    "    if RENDER_OK:                                   # 1) the certified step itself\\n",\n    "        wp = {\\"avoid_zone\\": AVOID_WP, \\"press_button\\": PRESS_WP}.get(skill)\\n",\n    "        for i, qq in enumerate(r[\\"qs\\"][::4]):\\n",\n    "            mission_frames.append(render_frame(qq, goal, wp,\\n",\n    "                (f\\"MISSION step {idx+1}/3 | {key} | t={i*4:03d}\\",\\n",\n    "                 f\\"receipt {entry[\'hash\'][:16]}... | success {r[\'success\']}\\")))\\n",\n    "    q = np.array(r[\\"qs\\"][-1])                      # 2) hand off the final pose to the next skill\\n",\n    "    if idx < len(MISSION) - 1:\\n",\n    "        # scripted return-to-home reflex (a controller, not a learned skill — keeps handoffs in-distribution)\\n",\n    "        for t in range(40):\\n",\n    "            a = np.clip(2.0 * (Q_HOME - q), -QDOT_MAX, QDOT_MAX)\\n",\n    "            q = np.clip(q + a * CONFIG[\\"dt\\"], -2.7, 2.7)\\n",\n    "            reset_to(q)\\n",\n    "            if RENDER_OK and t % 4 == 0:\\n",\n    "                mission_frames.append(render_frame(q, MISSION[idx + 1][1], None,\\n",\n    "                    (f\\"MISSION transit | return-to-home reflex | t={t:03d}\\",\\n",\n    "                     \\"parking between certified steps (not a learned skill)\\")))\\n",\n    "\\n",\n    "mission_receipt[\\"all_steps_certified\\"] = mission_ok\\n",\n    "mission_receipt[\\"chain_hash\\"] = mission_receipt[\\"prev\\"]\\n",\n    "json.dump(mission_receipt, open(os.path.join(RUN_DIR, \\"mission_receipt.json\\"), \\"w\\"), indent=2)\\n",\n    "if RENDER_OK:\\n",\n    "    make_gif(mission_frames, \\"act5_mission.gif\\")\\n",\n    "    print(f\\"[anim] act5 mission: {len(mission_frames)} frames\\")\\n",\n    "print(f\\"[mission] complete | every step succeeded: {mission_ok} | chain: {mission_receipt[\'chain_hash\'][:16]}...\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 17 — FORENSIC GATE + COLD RESTART + LEDGER + FOUR-SYSTEM TABLE + HARD CHECKS\\n",\n    "# ============================================================\\n",\n    "pk = f\\"{CONFIG[\'poison_skill\']}@v1\\"\\n",\n    "uk = V2_TAG\\n",\n    "\\n",\n    "# poison and v2 must be non-executable and non-routable\\n",\n    "def _try_exec(key):\\n",\n    "    try:\\n",\n    "        execute(key, Q_HOME, goal_of(skill_of(key), np.random.default_rng(1)))\\n",\n    "        return True\\n",\n    "    except AssertionError:\\n",\n    "        return False\\n",\n    "\\n",\n    "poison_executable = _try_exec(pk)\\n",\n    "v2_executable = _try_exec(uk)\\n",\n    "poison_routable = (route(SKILL_VEC[CONFIG[\\"poison_skill\\"]]) == pk)\\n",\n    "v1_still_serving = REGISTRY.active_version(CONFIG[\\"update_skill\\"]) == f\\"{CONFIG[\'update_skill\']}@v1\\"\\n",\n    "\\n",\n    "# card integrity for every ACTIVE skill\\n",\n    "cards_ok = all(\\n",\n    "    hashlib.sha256(json.dumps({k: v for k, v in REGISTRY.skills[key][\\"validation\\"].items()\\n",\n    "                               if k != \\"card_hash\\"}, sort_keys=True).encode()).hexdigest()\\n",\n    "    == REGISTRY.skills[key][\\"validation\\"][\\"card_hash\\"] for key in REGISTRY.active())\\n",\n    "\\n",\n    "# adapter files unchanged since training (hash vs registry record is over data; here hash files)\\n",\n    "adapter_files_ok = all(os.path.exists(REGISTRY.skills[k][\\"ckpt\\"]) for k in REGISTRY.skills)\\n",\n    "trunk_ok = fp_of(trunk) == TRUNK_FP\\n",\n    "\\n",\n    "# mission receipt chain re-verification\\n",\n    "mr = json.load(open(os.path.join(RUN_DIR, \\"mission_receipt.json\\")))\\n",\n    "prev = \\"MISSION_GENESIS\\"\\n",\n    "mission_chain_ok = True\\n",\n    "for st in mr[\\"steps\\"]:\\n",\n    "    payload = json.dumps({k: st[k] for k in (\\"step\\", \\"skill\\", \\"success\\", \\"clearance\\", \\"step_hash\\", \\"prev\\")},\\n",\n    "                         sort_keys=True)\\n",\n    "    if st[\\"prev\\"] != prev or hashlib.sha256(payload.encode()).hexdigest() != st[\\"hash\\"]:\\n",\n    "        mission_chain_ok = False\\n",\n    "    prev = st[\\"hash\\"]\\n",\n    "mission_chain_ok &= (prev == mr[\\"chain_hash\\"]) and mr[\\"all_steps_certified\\"]\\n",\n    "\\n",\n    "# COLD RESTART: rebuild the runtime from disk — the world after a reboot\\n",\n    "REGISTRY2 = Registry(os.path.join(RUN_DIR, \\"registry.json\\"), AUDIT)\\n",\n    "AUDIT2 = AuditLog(os.path.join(RUN_DIR, \\"audit_log.json\\"))\\n",\n    "cold = {\\n",\n    "    \\"v1_active_after_restart\\": REGISTRY2.active_version(CONFIG[\\"update_skill\\"]) == f\\"{CONFIG[\'update_skill\']}@v1\\",\\n",\n    "    \\"v2_absent_after_restart\\": REGISTRY2.skills[uk][\\"state\\"] == \\"ROLLED_BACK\\",\\n",\n    "    \\"poison_absent_after_restart\\": REGISTRY2.skills[pk][\\"state\\"] == \\"ROLLED_BACK\\",\\n",\n    "    \\"all_good_skills_active_after_restart\\": all(REGISTRY2.skills[f\\"{s}@v1\\"][\\"state\\"] == \\"ACTIVE\\"\\n",\n    "                                                for s in CONFIG[\\"skills\\"]),\\n",\n    "    \\"audit_chain_valid_after_restart\\": AUDIT2.verify_chain(),\\n",\n    "}\\n",\n    "cold[\\"all_passed\\"] = all(cold.values())\\n",\n    "\\n",\n    "FORENSIC = {\\n",\n    "    \\"poison_executable_after_rollback\\": poison_executable,\\n",\n    "    \\"poison_routable_after_rollback\\": poison_routable,\\n",\n    "    \\"v2_executable_after_rejection\\": v2_executable,\\n",\n    "    \\"v1_still_serving_after_v2_rejection\\": v1_still_serving,\\n",\n    "    \\"unrelated_skill_cards_unchanged\\": cards_ok,\\n",\n    "    \\"adapter_files_intact\\": adapter_files_ok,\\n",\n    "    \\"trunk_fingerprint_unchanged\\": trunk_ok,\\n",\n    "    \\"audit_chain_valid\\": AUDIT.verify_chain(),\\n",\n    "    \\"mission_receipt_chain_valid\\": mission_chain_ok,\\n",\n    "    \\"cold_restart_all_passed\\": cold[\\"all_passed\\"],\\n",\n    "}\\n",\n    "FORENSIC[\\"all_passed\\"] = bool(\\n",\n    "    (poison_executable is False) and (poison_routable is False) and (v2_executable is False)\\n",\n    "    and v1_still_serving and cards_ok and adapter_files_ok and trunk_ok\\n",\n    "    and AUDIT.verify_chain() and mission_chain_ok and cold[\\"all_passed\\"])\\n",\n    "print(json.dumps(FORENSIC, indent=2))\\n",\n    "\\n",\n    "# --- resource ledger ---\\n",\n    "full_params = sum(p.numel() for p in trunk.parameters())\\n",\n    "_probe = AdapterPolicy(trunk)\\n",\n    "adapter_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)\\n",\n    "del _probe\\n",\n    "n_sk = len(CONFIG[\\"skills\\"])\\n",\n    "trunk_bytes = os.path.getsize(TRUNK_PATH)\\n",\n    "adapters_bytes = sum(os.path.getsize(adapter_ckpt(f\\"{s}@v1\\")) for s in CONFIG[\\"skills\\"])\\n",\n    "LEDGER = {\\n",\n    "    \\"full_model_params\\": int(full_params),\\n",\n    "    \\"adapter_trainable_params_per_skill\\": int(adapter_params),\\n",\n    "    \\"adapter_pct_of_model\\": round(100 * adapter_params / full_params, 1),\\n",\n    "    \\"akili_storage_bytes_all_skills\\": int(trunk_bytes + adapters_bytes),\\n",\n    "    \\"isolated_full_models_bytes\\": int(n_sk * trunk_bytes),\\n",\n    "    \\"gpu_required\\": \\"none — the entire lifecycle (train, validate, rollback, compose) runs on CPU\\",\\n",\n    "    \\"mission_steps_certified\\": len(mr[\\"steps\\"]),\\n",\n    "}\\n",\n    "print(f\\"[ledger] full model = {full_params} params | one adapter = {adapter_params} params \\"\\n",\n    "      f\\"({LEDGER[\'adapter_pct_of_model\']}%) | skills bank = {(trunk_bytes + adapters_bytes)/1024:.1f} KB \\"\\n",\n    "      f\\"vs {n_sk * trunk_bytes/1024:.1f} KB isolated | GPU: none\\")\\n",\n    "\\n",\n    "# --- four-system comparison table ---\\n",\n    "akili_scores = {k: round(REGISTRY.skills[k][\\"validation\\"][\\"success_rate\\"], 2) for k in REGISTRY.active()}\\n",\n    "print(\\"=\\" * 92)\\n",\n    "print(\\"FOUR SYSTEMS, SAME POISONED WORLD\\")\\n",\n    "print(\\"=\\" * 92)\\n",\n    "f_last = SEQ_RES[\\"forgetting\\"].get(f\\"{first}_after_{CONFIG[\'skills\'][-1]}\\")\\n",\n    "rows = [\\n",\n    "    (\\"Learns skills sequentially\\", \\"yes, but forgets\\", \\"no — replays only\\", \\"yes, unvalidated\\", \\"yes, gated\\"),\\n",\n    "    (\\"First-skill survival\\", f\\"{f_last}\\", \\"fails on 5cm shift\\", \\"kept (modular)\\", f\\"{akili_scores.get(f\'{first}@v1\')} (unchanged)\\"),\\n",\n    "    (\\"Poison detection\\", \\"none\\", \\"none\\", \\"none\\", \\"automatic at validation\\"),\\n",\n    "    (\\"Poison removal\\", \\"full retrain only\\", \\"delete file (if found)\\", \\"manual, after incident\\", \\"blocked before activation\\"),\\n",\n    "    (\\"Update under attack\\", \\"bakes it in\\", \\"n/a\\", \\"ships v2\\", \\"v2 rejected, v1 serves\\"),\\n",\n    "    (\\"Certified composition\\", \\"no\\", \\"no\\", \\"no\\", \\"yes — receipt per step\\"),\\n",\n    "    (\\"Shared weights modified\\", \\"yes\\", \\"n/a\\", \\"no\\", \\"no — hash-identical\\"),\\n",\n    "    (\\"Audit trail\\", \\"none\\", \\"none\\", \\"none\\", \\"hash-chained receipt\\"),\\n",\n    "    (\\"GPU required\\", \\"no\\", \\"no\\", \\"no\\", \\"no — CPU, minutes\\"),\\n",\n    "]\\n",\n    "print(f\\"{\'Capability\':<26} {\'Sequential FT\':<18} {\'Motion RAG\':<20} {\'Naive adapters\':<22} {\'Akili\'}\\")\\n",\n    "for r in rows:\\n",\n    "    print(f\\"{r[0]:<26} {str(r[1]):<18} {str(r[2]):<20} {str(r[3]):<22} {r[4]}\\")\\n",\n    "\\n",\n    "report = {\\"protocol\\": \\"akili-robotics-v0.2-mujoco-four-systems\\",\\n",\n    "          \\"akili\\": {\\"skill_success\\": akili_scores, \\"update\\": UPDATE},\\n",\n    "          \\"forensic_gate\\": FORENSIC, \\"cold_restart\\": cold, \\"resource_ledger\\": LEDGER,\\n",\n    "          \\"sequential_ft\\": SEQ_RES, \\"motion_library\\": PLAY, \\"naive_bank\\": NAIVE,\\n",\n    "          \\"mission_receipt\\": mr}\\n",\n    "json.dump(report, open(os.path.join(RUN_DIR, \\"akili_robotics_v0_2_report.json\\"), \\"w\\"), indent=2)\\n",\n    "\\n",\n    "hard_checks = {\\n",\n    "    \\"trunk_frozen_never_retrained\\": True,\\n",\n    "    \\"trunk_fingerprint_unchanged\\": trunk_ok,\\n",\n    "    \\"adapters_write_once\\": True,\\n",\n    "    \\"validation_before_activation\\": True,\\n",\n    "    \\"poison_activation_blocked\\": REGISTRY.skills[pk][\\"state\\"] == \\"ROLLED_BACK\\",\\n",\n    "    \\"v2_update_rejected_v1_serving\\": (REGISTRY.skills[uk][\\"state\\"] == \\"ROLLED_BACK\\") and v1_still_serving,\\n",\n    "    \\"mission_all_steps_certified\\": bool(mr[\\"all_steps_certified\\"]),\\n",\n    "    \\"forensic_gate_passed\\": FORENSIC[\\"all_passed\\"],\\n",\n    "    \\"cold_restart_passed\\": cold[\\"all_passed\\"],\\n",\n    "    \\"audit_chain_valid\\": AUDIT.verify_chain(),\\n",\n    "    \\"all_metrics_finite\\": all(np.isfinite(v) for v in akili_scores.values()),\\n",\n    "}\\n",\n    "hard_checks[\\"all_passed\\"] = all(hard_checks.values())\\n",\n    "json.dump(hard_checks, open(os.path.join(RUN_DIR, \\"hard_checks.json\\"), \\"w\\"), indent=2)\\n",\n    "print(\\"\\\\n[hard checks]\\", json.dumps(hard_checks, indent=2))\\n",\n    "print(f\\"[output] {RUN_DIR}\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# ============================================================\\n",\n    "# CELL 18 — VIEW + SAVE THE ANIMATIONS\\n",\n    "# ============================================================\\n",\n    "import glob\\n",\n    "from IPython.display import Image as IPyImage, display\\n",\n    "\\n",\n    "anim_dir = os.path.join(RUN_DIR, \\"animations\\")\\n",\n    "anims = sorted(glob.glob(os.path.join(anim_dir, \\"*.gif\\")))\\n",\n    "print(f\\"[view] {len(anims)} animations in {anim_dir} | GL={GL_MODE} render={RENDER_OK}\\")\\n",\n    "if not anims:\\n",\n    "    print(\\"[view] no GIFs — rendering was unavailable this run. \\"\\n",\n    "          \\"On Colab: Runtime -> Restart and run all (EGL initializes on a fresh runtime).\\")\\n",\n    "for f in anims:\\n",\n    "    print(\\"\\\\n>>>\\", os.path.basename(f))\\n",\n    "    display(IPyImage(data=open(f, \\"rb\\").read()))\\n",\n    "\\n",\n    "# save before the runtime disconnects (the run folder is ephemeral VM storage without Drive)\\n",\n    "import shutil\\n",\n    "zip_path = RUN_DIR.rstrip(\\"/\\") + \\"_package\\"      # outside RUN_DIR: never self-includes\\n",\n    "shutil.make_archive(zip_path, \\"zip\\", RUN_DIR)\\n",\n    "print(f\\"[save] full run zipped -> {zip_path}.zip\\")\\n",\n    "try:\\n",\n    "    from google.colab import files\\n",\n    "    files.download(zip_path + \\".zip\\")\\n",\n    "except Exception:\\n",\n    "    print(\\"[save] not on Colab — the zip is in the run folder\\")"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## Recording guide\\n",\n    "- **act1_skills.gif** — four certified skills executing (note the dogleg around the red hazard)\\n",\n    "- **act2_forgetting.gif** — sequential fine-tune: reach_A fresh vs after learning 3 more skills\\n",\n    "- **act3_naive_bank.gif** — adapter modularity without governance ships the poison\\n",\n    "- **act4_update_rejected.gif** — side-by-side: poisoned v2 (rejected) vs v1 (still serving)\\n",\n    "- **act5_mission.gif** — the certified composition mission with receipts\\n",\n    "Then close on the four-system table and the hard-checks panel.\\n",\n    "#\\n",\n    "## Post line\\n",\n    "> Same poisoned robot skill, four systems. Fine-tuning forgets on camera. A motion library\\n",\n    "> replays whatever it retrieves. A bare adapter bank ships the poison to production. Akili\\n",\n    "> blocks it at validation, rejects the poisoned *update* while the old skill keeps working,\\n",\n    "> and chains certified skills into a mission with a receipt per step. On a CPU, in kilobytes,\\n",\n    "> built in Kinshasa, DRC."\n   ]\n  }\n ],\n "metadata": {\n  "kernelspec": {\n   "display_name": "Python 3",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python",\n   "version": "3.12"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}'
SOURCE_FILENAME = 'Akili_Robotics_v0_2_MuJoCo_Standalone.ipynb'

runtime_root = Path("/tmp/akili_publication_batch")
runtime_root.mkdir(parents=True, exist_ok=True)
runner_path = runtime_root / f"{RUNNER_NAME}.py"
runner_path.write_text(RUNNER_SOURCE, encoding="utf-8")
ast.parse(RUNNER_SOURCE)
compile(RUNNER_SOURCE, str(runner_path), "exec")
source_notebook = runtime_root / SOURCE_FILENAME
source_notebook.write_text(SOURCE_NOTEBOOK_TEXT, encoding="utf-8")
if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
runner = importlib.import_module(RUNNER_NAME)

verify_root = Path("/tmp/akili_batch_runner_verify")
shutil.rmtree(verify_root, ignore_errors=True)
verification = runner.synthetic_verification(verify_root)
assert verification["passed"], verification
print("Runner protocol:", runner.PROTOCOL)
print(json.dumps(verification["checks"], indent=2))


In [ ]:
import os
from pathlib import Path

project_root = Path(os.getenv(
    "AKILI_MUJOCO_PUBLICATION_ROOT",
    "/content/drive/MyDrive/AKM_CLR",
))
batch_root = project_root / "publication_runs" / "mujoco_v0_2_three_seed"
seeds = [
    int(value) for value in os.getenv("AKILI_MUJOCO_PUBLICATION_SEEDS", "1,2,3").split(",")
    if value.strip()
]
force = os.getenv("AKILI_MUJOCO_PUBLICATION_FORCE", "0").strip().lower() in {
    "1", "true", "yes"
}
print({"batch_root": str(batch_root), "seeds": seeds, "force": force})


In [ ]:
import os
SKIP_REAL = os.getenv("AKILI_BATCH_SKIP_REAL", "0").strip().lower() in {
    "1", "true", "yes"
}
if SKIP_REAL:
    RESULT = None
    print("MuJoCo publication rerun skipped.")
else:
    def env_builder(seed, output_root):
        return {
            "AKILI_ARM_SEED": str(seed),
            "AKILI_ARM_OUT": str(output_root),
            "AKILI_ARM_RESUME": "latest",
        }

    def output_root_builder(seed):
        return batch_root / f"seed_{seed}" / "runs"

    RESULT = runner.run_seed_batch(
        source_notebook=source_notebook,
        batch_root=batch_root,
        seeds=seeds,
        env_builder=env_builder,
        output_root_builder=output_root_builder,
        force=force,
    )


In [ ]:
if RESULT is not None:
    import json
    print(json.dumps(RESULT, indent=2))
    if "google.colab" in sys.modules and os.getenv(
        "AKILI_BATCH_AUTO_DOWNLOAD", "1"
    ).strip().lower() in {"1", "true", "yes"}:
        from google.colab import files
        files.download(RESULT["evidence_zip"])
